<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/ReidValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cow re-ID: validation v1 (Pen 3, 26 Sep, part 1 → part 2)

Follows `reid-pipeline-design.md` (v2). **The tracker only follows; it never names.** YOLO + ByteTrack give anonymous track pieces, and a separate identity layer names every piece from its look, or says "?" when unsure.

- **Gallery** = part 1 (`CalvingPen3Sep26Cow1.avi`): a person labels its tracks (cow A, B, …).
- **Test** = part 2 (`CalvingPen3Sep26Cow2.avi`): named by the system, from the gallery only. It is labelled separately, but those labels are **only used for scoring**.

Run the cells in order, on a T4 GPU, with `HF_TOKEN` in Colab Secrets.

| Cell | Step | Time (about) |
|---|---|---|
| 3 | Track + canonical crops, both parts, one pass each | 1–2 h per part |
| 3b | Preview video: boxes, TrackIds and SAM masks over 5 min | 3–5 min per part |
| 4 | 8-band DINOv3 fingerprints | 10–20 min per part |
| 5 | Labelling sheets + `Labels.csv` → **stop and label** | minutes |
| 6 | Tune the settings on part 1 only (frozen in `Tuned.json`) | minutes |
| 7 | Name part 2's tracks, and score them if part 2 is labelled | minutes |

A stopped cell 3 continues where it stopped when rerun. All outputs go to `ReidPipelineOutputs/<Run>/` on the Shared Drive.

In [ ]:
# 1. Setup: mount Drive, get the code from GitHub, install the libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"
if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from CalvingPipeline.Paths import FindVideo, ReidOutputRoot
from CalvingPipeline.Reid.Segments import TrackAndCrop, SeamCheck
from CalvingPipeline.Reid.Preview import PreviewVideo
from CalvingPipeline.Reid.Bands import EmbedBands
from CalvingPipeline.Reid.Labels import ReadLabels, SuggestGroups, TrackSheets, WriteLabelFile
from CalvingPipeline.Reid.Validate import Anchors, LoadVideo, NameVideo, ScoreNames, TuneOnGallery
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")
try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "NOT SET - cell 4 (DINOv3) will stop; add it in Colab Secrets")

In [ ]:
# 2. Settings: the only cell to edit.
Run = "Pen3Sep26"                                     # output folder: ReidPipelineOutputs/Pen3Sep26
Gallery = dict(VideoFile="CalvingPen3Sep26Cow1.avi", Name="Part1")   # labelled by a person -> anchors
Test = dict(VideoFile="CalvingPen3Sep26Cow2.avi", Name="Part2")      # named by the system, labels only for scoring
TryMinutes = None        # e.g. 20: only the first 20 min of each part (a quick test of cells 3-5); None = all

# --- Tracking and crops (cell 3) ---
TargetFps = 5            # frames per second tracked
EverySeconds = 15        # a canonical crop of every cow every 15 s
DetModel = "yolo26n.pt"  # COCO cow class; a fine-tuned model (cow / calf / person) can replace it later
ImageSize = 1024         # YOLO picture size
Confidence = 0.25        # lowest YOLO confidence kept
NativeFps = None         # None = the rate in the AVI header (150 for our cameras); times = chunk / NativeFps

# --- Crop gates (cells 5-7) ---
GateArea = 10000         # smallest usable mask (pixels); a cow far from the camera is ~15,000 in our 1920x1080 videos

# --- Labelling (cell 5) ---
SuggestThreshold = 0.70  # tracks this alike (and never seen together) are suggested as one cow

# --- Identity layer (cells 6-7): tuned on part 1 in cell 6, then frozen ---
Retune = False           # True = tune again (overwrites Tuned.json); never after looking at part 2's results
SeamMinutes = 10         # part 2's first minutes are scored apart (a cow standing still across the seam is easy)

# --- Folders (nothing to edit) ---
# --- Preview video (cell 3b) ---
PreviewMinutes = 5       # length of the preview clip
PreviewStart = None      # in seconds, e.g. 85 * 60; None = the 5 min with the most tracks
PreviewSpeed = 4         # played 4x faster

RunDir = ReidOutputRoot / Run
Dir = {V["Name"]: RunDir / V["Name"] for V in (Gallery, Test)}
EndSeconds = TryMinutes * 60 if TryMinutes else None
TrackSettings = dict(TargetFps=TargetFps, EverySeconds=EverySeconds, DetModel=DetModel, ImageSize=ImageSize,
                     Confidence=Confidence, NativeFps=NativeFps, EndSeconds=EndSeconds)
Gates = dict(GateArea=GateArea)
print("Outputs:", RunDir)

In [ ]:
# 3. Track + canonical crops (YOLO, ByteTrack, SAM 2.1 masks every 15 s), one pass per part. Rerun to continue.
for V in (Gallery, Test):
    print(f"\n=== {V['Name']}: {V['VideoFile']} ===")
    TrackAndCrop(FindVideo(V["VideoFile"]), Dir[V["Name"]], **TrackSettings)
print(SeamCheck(Dir[Gallery["Name"]], Dir[Test["Name"]], RunDir / "Seam.jpg"))
display(Image(str(RunDir / "Seam.jpg"), width=960))   # left: part 1's last frame, right: part 2's first

In [ ]:
# 3b. Preview video: YOLO + ByteTrack boxes with TrackIds and the SAM 2.1 masks, from cell 3's Tracks.csv.
#     Written to <part>/PreviewTracks.mp4 (overwritten). About 3-5 min per part for 5 min of video.
from IPython.display import Video
for V in (Gallery, Test):
    if (Dir[V["Name"]] / "Tracks.csv").exists():
        Clip = PreviewVideo(FindVideo(V["VideoFile"]), Dir[V["Name"]], StartSeconds=PreviewStart,
                            Minutes=PreviewMinutes, Speed=PreviewSpeed)
        display(Video(str(Clip), embed=True, width=960))


In [ ]:
# 4. 8-band fingerprints (DINOv3 ViT-S/16) of every crop. Skipped when up to date.
for V in (Gallery, Test):
    EmbedBands(Dir[V["Name"]], Token=HfToken)

In [ ]:
# 5. Labelling sheets. Then STOP and label: open <part>/Labels.csv (Drive: open with Google Sheets, then
#    File > Download > CSV, and upload it back under the same name), look at <part>/Sheets/Track<id>.jpg, and write
#    in "Cow": A, B, C ... (the same letter for the same cow in both parts), "new" (part 2 only: not in part 1),
#    X (not a cow, or two cows in one mask), or leave it empty if unsure. "Suggested" groups alike tracks.
#    Label part 1 first; label part 2 on your own, without looking at the system's names.
for V in (Gallery, Test):
    Crops, Bands, _ = LoadVideo(Dir[V["Name"]], **Gates)
    TrackSheets(Dir[V["Name"]], Crops)
    WriteLabelFile(Dir[V["Name"]], Crops, SuggestGroups(Crops, Bands, Threshold=SuggestThreshold))
    print(f"{V['Name']}: {Crops.TrackId.nunique()} tracks, {int(Crops.Gated.sum())} of {len(Crops)} crops usable, "
          f"{Crops[Crops.NonCow].TrackId.nunique()} tracks look coloured (staff?)")
display(Image(str(Dir[Gallery["Name"]] / "Sheets" / "Overview1.jpg"), width=1200))

In [ ]:
# 6. Tune the identity layer on part 1 ONLY (its first half names its second half), then freeze it.
TunedPath = RunDir / "Tuned.json"
GalleryCrops, GalleryBands, GalleryTracks = LoadVideo(Dir[Gallery["Name"]], **Gates)
GalleryLabels = ReadLabels(Dir[Gallery["Name"]])
print(f"Part 1: {len(GalleryLabels)} tracks labelled, cows {sorted(set(GalleryLabels.values()) - {'X', 'new'})}")
if TunedPath.exists() and not Retune:
    Tuned = json.loads(TunedPath.read_text())
    print("Frozen settings (Retune = True to tune again):", Tuned)
else:
    Tuned, Tried = TuneOnGallery(GalleryCrops, GalleryBands, GalleryTracks, GalleryLabels, CropSeconds=EverySeconds)
    TunedPath.write_text(json.dumps(Tuned, indent=1))
    Tried.to_csv(RunDir / "TuneOnPart1.csv", index=False)
    display(Tried.head(10))
    print("Chosen and frozen:", Tuned)

In [ ]:
# 7. Name part 2's tracks from part 1's labelled crops (anchors) + its own confident crops (lagged 30 min).
AnchorCrops, AnchorBands = Anchors(GalleryCrops, GalleryBands, GalleryLabels)
TestCrops, TestBands, TestTracks = LoadVideo(Dir[Test["Name"]], **Gates)
Names, CropNames, Windows = NameVideo(TestCrops, TestBands, TestTracks, AnchorCrops, AnchorBands,
                                      OutputDir=RunDir / "Names", **Tuned)
print("Track names:", Names.Name.value_counts().to_dict())
display(Names.head(20))
if (Dir[Test["Name"]] / "Labels.csv").exists() and ReadLabels(Dir[Test["Name"]]):
    Scores = ScoreNames(CropNames, ReadLabels(Dir[Test["Name"]]), sorted(AnchorCrops.Cow.unique()),
                        CropSeconds=EverySeconds, SeamMinutes=SeamMinutes, VideoStartSeconds=0)
    (RunDir / "Names" / "Scores.json").write_text(json.dumps(Scores, indent=1))
    display(pd.DataFrame(Scores).T)
else:
    print("Part 2 is not labelled yet: names written, nothing to score")

## About this notebook

Code: `CalvingPen/CalvingPipeline/Reid/` (`Frames.py`, `Canonical.py`, `Segments.py`, `Bands.py`, `Labels.py`, `Identity.py`, `Validate.py`). Parts are ported from [Sonam525/Re-id-for-tracking](https://github.com/Sonam525/Re-id-for-tracking) (MIT; see `Reid/NOTICE.md`).

**Outputs** in `ReidPipelineOutputs/<Run>/`:
- `<part>/Tracks.csv`, `Crops.csv`, `Crops/`, `Bands.npz`, `Sheets/`, `Labels.csv`, `Run.json`
- `Seam.jpg`, `Tuned.json`, `TuneOnPart1.csv`
- `Names/TrackNames.csv` (one name per track: a cow letter, "?" or "not a cow"), `CropNames.csv`, `Windows.csv` (every 3-min decision), `Scores.json`

**Scores** are in cow-minutes; each crop stands for 15 s of one cow:
- `Accuracy`: correct ÷ (correct + wrong) named time.
- `UnsureMinutes`: time called "?".
- Wrong time is split into silent errors and flagged errors (contested, low margin, short track).
- `NewCowRejection`: the share of time of cows not in part 1 that was called "?" or "new".

These are also given per night/day and for the first `SeamMinutes` vs the rest.

**Not in v1 yet:**
- a fine-tuned detector with calf and person classes (calves can still get in);
- BoviSync candidate cows and animalIds;
- an overlay video;
- splitting a track when ByteTrack swaps two cows.